# LeetCode #44: Wildcard Matching

https://leetcode.com/problems/wildcard-matching/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(2^{m+n})$ | $O(m+n)$ |
| **Optimal: 1D DP ★** | $O(m \times n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
Recursively match characters, branching on every `'*'` to try matching zero or more characters. Without memoization the branching is exponential.

### Optimal: 1D DP ★
`dp[j]` = whether `s[0..j-1]` matches `p[0..i-1]`. Process pattern row by row: `'?'` shifts a match diagonally; `'*'` propagates from the left (match zero chars) or keeps the current value (match one more char). Using a rolling 1D array reduces space from $O(mn)$ to $O(n)$.

**Constraints:**
* $0 \leq s.\text{length} \leq 2000$
* $0 \leq p.\text{length} \leq 2000$
* `p` contains only lowercase letters, `'?'`, and `'*'`


## Solutions

### C#

In [ ]:
public class Solution {
    public bool IsMatch(string s, string p) {
        int m = s.Length, n = p.Length;
        // dp[j] = true if s[0..j-1] matches the current pattern prefix
        bool[] dp = new bool[n + 1];
        dp[0] = true; // Empty string matches empty pattern

        // Leading stars can match the empty string
        for (int j = 1; j <= n; j++)
            dp[j] = dp[j - 1] && p[j - 1] == '*';

        for (int i = 1; i <= m; i++) {
            bool prev = dp[0]; // Holds dp[j-1] from the previous row
            dp[0] = false;     // Non-empty string cannot match empty pattern

            for (int j = 1; j <= n; j++) {
                bool temp = dp[j];
                if (p[j - 1] == '*') {
                    // '*' matches zero chars (dp[j-1]) or one more char (dp[j] from prev row)
                    dp[j] = dp[j - 1] || dp[j];
                } else {
                    // '?' or exact letter: inherit diagonal match from previous row/column
                    dp[j] = prev && (p[j - 1] == '?' || p[j - 1] == s[i - 1]);
                }
                prev = temp;
            }
        }

        return dp[n];
    }
}

### Python

In [ ]:
class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        m, n = len(s), len(p)
        # dp[j] = True if s[0..j-1] matches the current pattern prefix
        dp = [False] * (n + 1)
        dp[0] = True  # Empty string matches empty pattern

        # Leading stars can match the empty string
        for j in range(1, n + 1):
            dp[j] = dp[j - 1] and p[j - 1] == '*'

        for i in range(1, m + 1):
            prev = dp[0]  # Holds dp[j-1] from the previous row
            dp[0] = False  # Non-empty string cannot match empty pattern

            for j in range(1, n + 1):
                temp = dp[j]
                if p[j - 1] == '*':
                    # '*' matches zero chars (dp[j-1]) or one more char (dp[j] from prev row)
                    dp[j] = dp[j - 1] or dp[j]
                else:
                    # '?' or exact letter: inherit diagonal match
                    dp[j] = prev and (p[j - 1] == '?' or p[j - 1] == s[i - 1])
                prev = temp

        return dp[n]


### Go

In [ ]:
func isMatch(s string, p string) bool {
    m, n := len(s), len(p)
    // dp[j] = true if s[0..j-1] matches the current pattern prefix
    dp := make([]bool, n+1)
    dp[0] = true // Empty string matches empty pattern

    // Leading stars can match the empty string
    for j := 1; j <= n; j++ {
        dp[j] = dp[j-1] && p[j-1] == '*'
    }

    for i := 1; i <= m; i++ {
        prev := dp[0] // Holds dp[j-1] from the previous row
        dp[0] = false // Non-empty string cannot match empty pattern

        for j := 1; j <= n; j++ {
            temp := dp[j]
            if p[j-1] == '*' {
                // '*' matches zero chars (dp[j-1]) or one more char (dp[j] from prev row)
                dp[j] = dp[j-1] || dp[j]
            } else {
                // '?' or exact letter: inherit diagonal match
                dp[j] = prev && (p[j-1] == '?' || p[j-1] == s[i-1])
            }
            prev = temp
        }
    }

    return dp[n]
}

### Rust

In [ ]:
impl Solution {
    pub fn is_match(s: String, p: String) -> bool {
        let s_bytes = s.as_bytes();
        let p_bytes = p.as_bytes();
        let (m, n) = (s.len(), p.len());

        // dp[j] = true if s[0..j-1] matches the current pattern prefix
        let mut dp = vec![false; n + 1];
        dp[0] = true; // Empty string matches empty pattern

        // Leading stars can match the empty string
        for j in 1..=n {
            dp[j] = dp[j - 1] && p_bytes[j - 1] == b'*';
        }

        for i in 1..=m {
            let mut prev = dp[0]; // Holds dp[j-1] from the previous row
            dp[0] = false; // Non-empty string cannot match empty pattern

            for j in 1..=n {
                let temp = dp[j];
                dp[j] = if p_bytes[j - 1] == b'*' {
                    // '*' matches zero chars (dp[j-1]) or one more char (dp[j] from prev row)
                    dp[j - 1] || dp[j]
                } else {
                    // '?' or exact letter: inherit diagonal match
                    prev && (p_bytes[j - 1] == b'?' || p_bytes[j - 1] == s_bytes[i - 1])
                };
                prev = temp;
            }
        }

        dp[n]
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `s = "aa", p = "a*"`
`'*'` matches the second `'a'`, so the result is `true`. The DP propagates `dp[2] = true` when the `'*'` column persists from the previous row.

### 2. Slightly Complex
**Input:** `s = "cb", p = "?a"`
`'?'` matches `'c'`, but `'a'` does not match `'b'` — result is `false`. The diagonal propagation places `false` at `dp[2]` in the final row.

### 3. Edge Case: Time Factor
**Input:** `s = "aaaaaa...a"` (2000 `a`s), `p = "*a*a*a"` (alternating stars and letters)
The DP fills all $2000 \times 2000 = 4 \times 10^6$ cells — the worst-case time path — but terminates correctly.

### 4. Edge Case: Space Factor
**Input:** `s = "abcdef"` (length 6), `p = "*"`
A single `'*'` matches any string. The DP array has length 2 (`n+1=2`), but the single `'*'` column stays `true` through all rows — $O(n)$ is as small as $O(1)$ here.

### 5. Almost-Impossible but Plausible
**Input:** `s = "aab", p = "c*a*b"`
Pattern starts with `'c'`, which doesn't match `'a'`, so `dp[1]` goes `false` in row 1 and the mismatch propagates — result `false`. The early-false propagation shows how `'*'`-heavy patterns still correctly reject mismatches.
